# NLP 作業3

## 安裝需要的套件

In [53]:
!pip install datasets==2.19.0
!pip install evaluate
!pip install seqeval

In [54]:
import datasets
import evaluate
import seqeval

print(datasets.__version__)
print(evaluate.__version__)

2.19.0
0.4.6


In [55]:
! pip list | grep seqeval

seqeval                                  1.2.2


In [56]:
import torch

# 測試現在這個 Colab 環境是否已經使用 GPU
# 否則等下可能會需要重新啟動 Colab 環境
torch.cuda.is_available() # 結果需要顯示為 True

True

## Start

In [57]:
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import EvalPrediction

In [58]:
MODEL_NAME = "bert-base-uncased"
DATA_NAME = "ncbi_disease"
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 20
NUM_TRAIN_EPOCHS = 10
LEARNING_RATE = 3e-5

In [59]:
# 載入資料集
dataset = datasets.load_dataset(DATA_NAME, trust_remote_code=True)

In [60]:
# 檢查每個 split 的數量

for data_type in ["train", "validation", "test"]:
    print(f"{data_type}: {len(dataset[data_type])} samples")

train: 5433 samples
validation: 924 samples
test: 941 samples


In [61]:
# Show NER tag names
label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)

['O', 'B-Disease', 'I-Disease']


In [62]:
# 觀察資料
first_example = dataset["train"][1]
print(type(first_example))

for k, v in first_example.items():
    print(f"{k}: {v}")

<class 'dict'>
id: 1
tokens: ['The', 'adenomatous', 'polyposis', 'coli', '(', 'APC', ')', 'tumour', '-', 'suppressor', 'protein', 'controls', 'the', 'Wnt', 'signalling', 'pathway', 'by', 'forming', 'a', 'complex', 'with', 'glycogen', 'synthase', 'kinase', '3beta', '(', 'GSK', '-', '3beta', ')', ',', 'axin', '/', 'conductin', 'and', 'betacatenin', '.']
ner_tags: [0, 1, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [63]:
# TODO1: 建立 tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [64]:
tokenized_inputs = tokenizer(
        first_example["tokens"],
        truncation=True,
        is_split_into_words=True,
)
print("Tokenized 後的結果：")
print(tokenized_inputs)
print("Tokenized 後的 word_ids：")
print(tokenized_inputs.word_ids())

print("原始資料的 labels：")
print(first_example["ner_tags"])
print("我們下一步需要轉換 labels 為：")
print("[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, -100]")

Tokenized 後的結果：
{'input_ids': [101, 1996, 16298, 9626, 24826, 2015, 26572, 6873, 6190, 27441, 1006, 9706, 2278, 1007, 10722, 20360, 1011, 16081, 2953, 5250, 7711, 1996, 1059, 3372, 21919, 12732, 2011, 5716, 1037, 3375, 2007, 1043, 2135, 3597, 6914, 24203, 11022, 21903, 1017, 20915, 2050, 1006, 28177, 2243, 1011, 1017, 20915, 2050, 1007, 1010, 22260, 2378, 1013, 6204, 2378, 1998, 8247, 16280, 11483, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Tokenized 後的 word_ids：
[None, 0, 1, 1, 1, 1, 2, 2, 2, 3, 4, 5, 5, 6, 7, 7, 8, 9, 9, 10, 11, 12, 13, 13, 14, 15, 16, 17, 18, 19, 20, 21, 21, 21, 21, 22, 22, 23, 24, 24, 24, 25, 26, 

In [65]:
# TODO2: labels 處理
# 我們可以發現資料集的 labels (ner_tags) 是以 word 為單位
# 但是我們要使用的 tokenizer (如 BERT 的 tokenizer) 會把 word 切成 subwords
# 因此要針對 labels 的部分進行處理

def tokenize_and_align_labels(example):
    original_labels = example["ner_tags"]

    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True,
    )
    word_ids = tokenized_inputs.word_ids()
    labels = []
    current_word_idx = None
    for word_idx in word_ids:
        # Write your code here
        # Hints:
        # (1) [CLS] or [SEP] 設為 -100
        # (2) 由左至右給予新的 labels，
        # 因此需要 current_word_idx
        # 來幫助我們觀察下個 token 是否進到新的 word，還是是 上一個 word 的 sub-word

        # (1) [CLS] 或 [SEP] 或 Padding 設為 -100
        if word_idx is None:
            labels.append(-100)
        # (2) 如果進入了新單詞，給予該單詞的原始標籤
        elif word_idx != current_word_idx:
            labels.append(original_labels[word_idx])
        # (3) 如果是同一個單詞的 sub-word，設為 -100 以避免重複計算標籤
        else:
            labels.append(-100)

        current_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [66]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=False,
    remove_columns=dataset["train"].column_names,
)

In [67]:
tokenized_datasets["train"]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5433
})

In [68]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [69]:
fake_batch = [tokenized_datasets["train"][i] for i in range(2)]
batch = data_collator(fake_batch)
batch["labels"]

tensor([[-100,    0,    0,    0, -100, -100,    0,    0,    0, -100,    0,    0,
            1, -100, -100, -100,    2, -100, -100,    2,    2, -100,    0, -100,
            0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100],
        [-100,    0,    1, -100, -100, -100,    2, -100, -100,    2,    2,    2,
         -100,    2,    2, -100,    0,    0, -100,    0,    0,    0,    0, -100,
            0,    0,    0,    0,    0,    0,    0,    0, -100, -100, -100,    0,
         -100,    0,    0, -100, -100,    0,    0, -100,    0,    0, -100, -100,
            0,    0,    0, -100,    0,    0, -100,    0,    0, -100, -100,    0,
         -100]])

In [70]:
# TODO3: 建立 id -> label (`id2label`) 以及 label -> id (`label2id`) 的轉換
# `id2label` 和 `label2id` 都是 Python dict，且 key 跟 value 都是 int

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

In [71]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [72]:
# 檢查模型是否確實被設定為3種類別輸出
model.config.num_labels

3

In [73]:
# TODO4: 設定 TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=0.01,
    report_to='tensorboard',
    push_to_hub=False,
)

In [74]:
# TODO5: 完成 compute_metrics

metric = evaluate.load("seqeval")

def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # 將預測與真實標籤轉換回文字，並忽略標籤為 -100 的位置
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [75]:
# TODO6: set up trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

### 相關官方文件連結
- [TokenClassifierOutput](https://huggingface.co/docs/transformers/en/main_classes/output#transformers.modeling_outputs.TokenClassifierOutput)
- [EvalPrediction](https://huggingface.co/docs/transformers/internal/trainer_utils#transformers.EvalPrediction)

In [76]:
# compute_metrics 模擬測試區
import numpy as np  # 加上這一行
eval_dataloader = trainer.get_eval_dataloader()
batch = next(iter(eval_dataloader))
with torch.no_grad():
    outputs = model(**{k: v.to(model.device) for k, v in batch.items()})
print(f"AutoModelForTokenClassification 輸出的型態為: {type(outputs)}")
print(f"Logits shape: {outputs.logits.shape}")
print(f"Labels shape: {batch['labels'].shape}")
print(f"Loss: {outputs.loss.item()}")
print(f"Type of `outputs.loss`: {type(outputs.loss)}")
print(f"Type of `outputs.loss.item()`: {type(outputs.loss.item())}")
print()

# 取得 logits 和 labels
logits = outputs.logits.cpu().numpy()
labels = batch["labels"].cpu().numpy()

# 建立 EvalPrediction 模擬 compute_metrics 呼叫
mock_eval = EvalPrediction(
    predictions=logits,
    label_ids=labels,
)
print(f"Trainer 在輸進去 compute_metrics 前的型態為: {type(mock_eval)}")

# 呼叫你自己寫的 metrics function
metrics = compute_metrics(mock_eval)
for k, v in metrics.items():
    print(f"{k}: {v}")

AutoModelForTokenClassification 輸出的型態為: <class 'transformers.modeling_outputs.TokenClassifierOutput'>
Logits shape: torch.Size([20, 59, 3])
Labels shape: torch.Size([20, 59])
Loss: 1.076918125152588
Type of `outputs.loss`: <class 'torch.Tensor'>
Type of `outputs.loss.item()`: <class 'float'>

Trainer 在輸進去 compute_metrics 前的型態為: <class 'transformers.trainer_utils.EvalPrediction'>
precision: 0.012295081967213115
recall: 0.1875
f1: 0.023076923076923075
accuracy: 0.4077079107505071


In [77]:
trainer.train()

Step,Training Loss
500,0.071254
1000,0.018966
1500,0.006728
2000,0.002443
2500,0.001335
3000,0.000732


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3400, training_loss=0.014991725174819722, metrics={'train_runtime': 205.1319, 'train_samples_per_second': 264.854, 'train_steps_per_second': 16.575, 'total_flos': 1920408689557692.0, 'train_loss': 0.014991725174819722, 'epoch': 10.0})

In [78]:
trainer.evaluate(tokenized_datasets["test"])

{'eval_loss': nan,
 'eval_precision': 0.8269040553907022,
 'eval_recall': 0.8708333333333333,
 'eval_f1': 0.8483003551496702,
 'eval_accuracy': 0.9822835449238683,
 'eval_runtime': 1.5566,
 'eval_samples_per_second': 604.533,
 'eval_steps_per_second': 30.837,
 'epoch': 10.0}

In [79]:
import sys
import torch
import datasets
import evaluate
import transformers
import numpy as np

# 整理套件資訊
# 註：seqeval 通常透過 pip list 檢查，此處手動加入你作業範例中的版本
packages = [
    ("Python", sys.version.split()[0]),
    ("torch (PyTorch)", torch.__version__),
    ("transformers", transformers.__version__),
    ("datasets", datasets.__version__),
    ("evaluate", evaluate.__version__),
    ("numpy", np.__version__),
    ("seqeval", "1.2.2"),
]

# 輸出對齊表格
print(f"{'項目 (Package)':<25} | {'版本 (Version)':<15}")
print("-" * 45)
for pkg, ver in packages:
    print(f"{pkg:<25} | {ver:<15}")

項目 (Package)              | 版本 (Version)   
---------------------------------------------
Python                    | 3.12.13        
torch (PyTorch)           | 2.10.0+cu128   
transformers              | 5.0.0          
datasets                  | 2.19.0         
evaluate                  | 0.4.6          
numpy                     | 2.0.2          
seqeval                   | 1.2.2          
